# 1 FEATURE , PAST 10 YEAR , NEXT DAY PREDICTION WINDOW SIZE=6

In [1]:
!pip install meteostat

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.7 MB 5.6 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/9.7 MB 6.6 MB/s eta 0:00:02
   ----------------- ---------------------- 4.2/9.7 MB 6.8 MB/s eta 0:00:01
   -------------------- ------------------- 5.0/9.7 MB 5.9 MB/s eta 0:00:01
   ----------------------- ---------------- 5.8/9.7 MB 5.5 MB/s eta 0:00:01
   ------------------------- -------------- 6.3/9.7 MB 5.2 MB/s eta 0:00:01
   --------------------------- ------------ 6.8/9.7 MB 4.8 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.7 MB 4.4 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.7 MB 4.2 MB/s eta 0:00:01
   -------------------------------- ------- 7.9/9.7 MB 3.9 MB/s eta 0:00:01
   --------------------------------- ------ 8.1/9.7 MB 3.6 MB/s eta 0:00:01
   ---------------------------------- ----- 8.4/9.7 MB 3.4 MB/s eta 0:00:01
   ----------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
erddapy 2.2.4 requires pandas<3,>=0.25.2, but you have pandas 3.0.1 which is incompatible.
gradio 5.29.1 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
mlflow 3.9.0 requires pandas<3, but you have pandas 3.0.1 which is incompatible.
streamlit 1.50.0 requires pandas<3,>=1.4.0, but you have pandas 3.0.1 which is incompatible.


In [ ]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from geopy.geocoders import Nominatim
from meteostat import Point, Daily
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
geolocator = Nominatim(user_agent='jainam')
city = 'Ahmedabad, India'
location = geolocator.geocode(city)

if location:
    print(f'City: {city}')
    print(f'Latitude: {location.latitude}, Longitude: {location.longitude}')

In [ ]:
location_point = Point(23.0215374, 72.5800568)
start = datetime.datetime(2014, 6, 1)
end = datetime.datetime(2024, 6, 1)

data = Daily(location_point, start, end).fetch()
temperature_avg = data[['tavg']].copy()
temperature_avg.dropna(inplace=True)
temperature_avg

In [ ]:
temperature_avg_seq = temperature_avg.values
len(temperature_avg_seq)

In [ ]:
def plot_loss_curves(history):
    loss = history['loss']
    val_loss = history['val_loss']
    mae = history['mae']
    val_mae = history['val_mae']

    epochs = range(len(loss))

    plt.plot(epochs, loss, label='training_loss')
    plt.plot(epochs, val_loss, label='val_loss')
    plt.title('Loss')
    plt.xlabel('Epochs')
    plt.legend()

    plt.figure()
    plt.plot(epochs, mae, label='training_mae')
    plt.plot(epochs, val_mae, label='val_mae')
    plt.title('MAE')
    plt.xlabel('Epochs')
    plt.legend()

In [ ]:
window_size = 6
X = []
Y = []

for i in range(len(temperature_avg_seq) - window_size):
    X.append(temperature_avg_seq[i:i + window_size])
    Y.append(temperature_avg_seq[i + window_size])

X = np.array(X)
Y = np.array(Y)

In [ ]:
X_scaler = MinMaxScaler()
Y_scaler = MinMaxScaler()

X_flat = X.reshape(-1, 1)
Y_flat = Y.reshape(-1, 1)

X_scaled = X_scaler.fit_transform(X_flat).reshape(len(X), window_size, 1)
Y_scaled = Y_scaler.fit_transform(Y_flat).reshape(len(Y), 1)

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X_scaled, Y_scaled, test_size=0.2, random_state=42, shuffle=True
)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
Y_train_t = torch.tensor(Y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
Y_val_t = torch.tensor(Y_val, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, Y_train_t), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, Y_val_t), batch_size=16, shuffle=False)

In [ ]:
class NaiveLSTM(nn.Module):
    def __init__(self, input_dim=1):
        super().__init__()
        self.lstm1 = nn.LSTM(input_dim, 128, batch_first=True)
        self.dropout1 = nn.Dropout(0.2)
        self.lstm2 = nn.LSTM(128, 64, batch_first=True)
        self.dropout2 = nn.Dropout(0.2)
        self.lstm3 = nn.LSTM(64, 32, batch_first=True)
        self.dropout3 = nn.Dropout(0.2)
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        x, _ = self.lstm1(x)
        x = self.dropout1(x)
        x, _ = self.lstm2(x)
        x = self.dropout2(x)
        x, _ = self.lstm3(x)
        x = self.dropout3(x)
        x = x[:, -1, :]
        x = self.fc(x)
        return x

naive_model = NaiveLSTM().to(device)
naive_model

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(naive_model.parameters(), lr=0.001)

history = {'loss': [], 'val_loss': [], 'mae': [], 'val_mae': []}

for _ in range(100):
    naive_model.train()
    train_losses = []
    train_maes = []

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        preds = naive_model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_maes.append(torch.mean(torch.abs(preds - yb)).item())

    naive_model.eval()
    val_losses = []
    val_maes = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = naive_model(xb)
            vloss = criterion(preds, yb)
            val_losses.append(vloss.item())
            val_maes.append(torch.mean(torch.abs(preds - yb)).item())

    history['loss'].append(float(np.mean(train_losses)))
    history['val_loss'].append(float(np.mean(val_losses)))
    history['mae'].append(float(np.mean(train_maes)))
    history['val_mae'].append(float(np.mean(val_maes)))

In [ ]:
plot_loss_curves(history)

In [ ]:
future_start = datetime.datetime(2024, 6, 1)
future_end = datetime.datetime(2024, 12, 1)
future_data = Daily(location_point, future_start, future_end).fetch()
next_6_months_temp = future_data['tavg'].dropna().values
plt.plot(next_6_months_temp)

In [ ]:
seed_start = datetime.datetime(2024, 5, 26)
seed_end = datetime.datetime(2024, 5, 31)
seed_data = Daily(location_point, seed_start, seed_end).fetch()
seed_tavg = seed_data['tavg'].dropna().values.reshape(-1, 1)

seed_scaled = X_scaler.transform(seed_tavg).reshape(-1)
data_for_model = seed_scaled.copy()

In [ ]:
output = []
steps = len(next_6_months_temp)

naive_model.eval()
for _ in range(steps):
    inp = torch.tensor(data_for_model.reshape(1, window_size, 1), dtype=torch.float32, device=device)
    with torch.no_grad():
        prediction_scaled = naive_model(inp).cpu().numpy().reshape(1, 1)

    prediction_original = Y_scaler.inverse_transform(prediction_scaled)[0, 0]
    output.append(prediction_original)

    next_value_scaled = prediction_scaled.flatten()[0]
    data_for_model = np.append(data_for_model[1:], next_value_scaled)

In [ ]:
plt.plot(list(next_6_months_temp), label='Actual Temperature')
plt.plot(list(output), label='Predicted Temperature')
plt.legend()
plt.xlabel('Days')
plt.ylabel('Temperature')
plt.title('Actual vs Predicted Temperature')
plt.show()

In [ ]:
r2_score(list(next_6_months_temp), output)

In [ ]:
torch.save(naive_model.state_dict(), './naive_model.pth')

# EVALUATION

In [ ]:
def model_evaluation(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    medianae = median_absolute_error(y_true, y_pred)
    metrics = {
        'Mean Squared Error': mse,
        'Mean Absolute Error': mae,
        'Median Absolute Error': medianae
    }
    return metrics

In [ ]:
def display_metrics(metrics, save_path=None):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(list(metrics.keys()), list(metrics.values()), color='skyblue')
    ax.set_title('Model Evaluation Metrics', fontsize=16, fontweight='bold')
    ax.set_xlabel('Scores', fontsize=12)

    for i, v in enumerate(metrics.values()):
        ax.text(v + 0.01, i, f'{v:.2f}', color='black', va='center', fontsize=10)

    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()

In [ ]:
metrics = model_evaluation(list(next_6_months_temp), output)
display_metrics(metrics, 'Matrix_Evaluation_avg.png')
metrics